# Titanic Dataset — Data Cleaning & Exploratory Data Analysis

Task 2 of Data Science Virtual Internship.

Goal: Clean the Titanic dataset and explore relationships between passenger variables and survival.

## Step 1: Load & Inspect Data

In [ ]:
import pandas as pd

df = pd.read_csv("train.csv")

print("Shape:", df.shape)
print("\nColumn info:")
print(df.info())

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nSummary statistics:")
print(df.describe())


## Step 2: Data Cleaning

- **Age**: filled missing values using median grouped by `Pclass` + `Sex`
- **Embarked**: filled missing values with the mode (most frequent port)
- **Cabin**: too sparse (77% missing) to impute — converted into a binary `HasCabin` feature instead
- Dropped identifier/free-text columns (`PassengerId`, `Ticket`, `Name`)
- Engineered `FamilySize` and `IsAlone` features

In [ ]:
import pandas as pd

df = pd.read_csv("train.csv")

# 1. Age: 177 missing (~20%) - fill with MEDIAN age grouped by Pclass + Sex
#    (more accurate than a single overall median, since age varies by class/gender)
df["Age"] = df.groupby(["Pclass", "Sex"])["Age"].transform(lambda x: x.fillna(x.median()))

# 2. Embarked: only 2 missing - fill with the MODE (most common port)
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# 3. Cabin: 687 missing (77%) - too sparse to fill meaningfully.
#    Instead of dropping the column entirely, engineer a new binary feature:
#    "HasCabin" (1 if cabin info exists, 0 if not) - cabin data presence
#    itself may correlate with class/fare/survival.
df["HasCabin"] = df["Cabin"].notnull().astype(int)
df.drop(columns=["Cabin"], inplace=True)

# 4. Drop columns not useful for pattern analysis (unique identifiers/free text)
df.drop(columns=["PassengerId", "Ticket", "Name"], inplace=True)

# 5. Feature engineering: Family size = siblings/spouses + parents/children + self
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

print("Missing values after cleaning:")
print(df.isnull().sum())
print("\nCleaned shape:", df.shape)
print("\nSample rows:")
print(df.head())

df.to_csv("train_cleaned.csv", index=False)
print("\nSaved cleaned data to train_cleaned.csv")


## Step 3: Exploratory Data Analysis

Visualizing relationships between passenger attributes and survival.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("train_cleaned.csv")
sns.set_style("whitegrid")

# --- Chart 1: Survival rate by Sex ---
plt.figure(figsize=(6,5))
survival_by_sex = df.groupby("Sex")["Survived"].mean()
ax = survival_by_sex.plot(kind="bar", color=["#e74c3c","#3498db"])
plt.title("Survival Rate by Sex", fontsize=13, fontweight="bold")
plt.ylabel("Survival Rate")
plt.xticks(rotation=0)
for i, v in enumerate(survival_by_sex):
    ax.text(i, v + 0.02, f"{v:.0%}", ha="center", fontweight="bold")
plt.tight_layout()
plt.savefig("chart1_survival_by_sex.png", dpi=150)
plt.close()

# --- Chart 2: Survival rate by Passenger Class ---
plt.figure(figsize=(6,5))
survival_by_class = df.groupby("Pclass")["Survived"].mean()
ax = survival_by_class.plot(kind="bar", color="#2ecc71")
plt.title("Survival Rate by Passenger Class", fontsize=13, fontweight="bold")
plt.xlabel("Passenger Class (1=Upper, 3=Lower)")
plt.ylabel("Survival Rate")
plt.xticks(rotation=0)
for i, v in enumerate(survival_by_class):
    ax.text(i, v + 0.02, f"{v:.0%}", ha="center", fontweight="bold")
plt.tight_layout()
plt.savefig("chart2_survival_by_class.png", dpi=150)
plt.close()

# --- Chart 3: Age distribution (Survived vs Not) ---
plt.figure(figsize=(8,5))
sns.histplot(data=df, x="Age", hue="Survived", multiple="stack", bins=30, palette=["#e74c3c","#2ecc71"])
plt.title("Age Distribution by Survival", fontsize=13, fontweight="bold")
plt.xlabel("Age")
plt.legend(title="Survived", labels=["Yes","No"])
plt.tight_layout()
plt.savefig("chart3_age_distribution.png", dpi=150)
plt.close()

# --- Chart 4: Fare distribution (Survived vs Not) - boxplot ---
plt.figure(figsize=(6,5))
sns.boxplot(data=df, x="Survived", y="Fare", palette=["#e74c3c","#2ecc71"])
plt.title("Fare Distribution by Survival", fontsize=13, fontweight="bold")
plt.xlabel("Survived (0=No, 1=Yes)")
plt.ylabel("Fare")
plt.ylim(0, 300)  # remove extreme outliers for readability
plt.tight_layout()
plt.savefig("chart4_fare_by_survival.png", dpi=150)
plt.close()

# --- Chart 5: Correlation heatmap ---
plt.figure(figsize=(8,6))
numeric_df = df.select_dtypes(include=["int64","float64"])
corr = numeric_df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap of Numeric Features", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("chart5_correlation_heatmap.png", dpi=150)
plt.close()

# --- Chart 6: Survival rate by Family Size ---
plt.figure(figsize=(7,5))
survival_by_family = df.groupby("FamilySize")["Survived"].mean()
ax = survival_by_family.plot(kind="bar", color="#9b59b6")
plt.title("Survival Rate by Family Size", fontsize=13, fontweight="bold")
plt.xlabel("Family Size (incl. self)")
plt.ylabel("Survival Rate")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("chart6_survival_by_familysize.png", dpi=150)
plt.close()

print("All charts saved.")

# Print key numeric findings for the README
print("\n--- KEY FINDINGS ---")
print("Overall survival rate:", round(df["Survived"].mean()*100,1), "%")
print("\nSurvival by Sex:\n", survival_by_sex)
print("\nSurvival by Pclass:\n", survival_by_class)
print("\nSurvival by FamilySize:\n", survival_by_family)
print("\nCorrelation with Survived:\n", corr["Survived"].sort_values(ascending=False))


## Key Findings

- **Sex**: Women survived at 74%, men at only 19% — the strongest single predictor of survival.
- **Passenger Class**: 1st class passengers survived at 63%, vs only 24% in 3rd class.
- **Age**: Younger passengers had a higher survival share.
- **Fare**: Survivors paid noticeably higher fares on average (tied to class).
- **Family Size**: Small families (2-4 members) survived more than solo travelers or very large families.
- **Correlation**: `Pclass` (-0.34) and `HasCabin` (+0.32) were the strongest numeric correlates with survival.

## Conclusion
Survival on the Titanic was strongly driven by **sex**, **socio-economic class**, and to a lesser extent **age** and **family size** — consistent with the historical "women and children first" evacuation protocol and the advantage wealthier passengers had in accessing lifeboats.